Code chạy Zero-shot (B)

In [1]:
import sys, subprocess
print(sys.executable)
r = subprocess.run([sys.executable, "-m", "pip", "install", "tiktoken"],
                   capture_output=True, text=True)
print(r.stdout[-500:]); print(r.stderr[-500:])

d:\uit-viquad2-extractive-qa\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\venv\Scripts\python.exe
ring_on_UIT_ViQuAD_2.0_Project\venv\Lib\site-packages (from requests->tiktoken) (3.20)




In [2]:
import sys, json, re, string, unicodedata
from pathlib import Path
from collections import Counter
import transformers
import numpy as np
import torch
from transformers import pipeline
import tiktoken

ROOT = Path.cwd().parent
DEVICE = 0 if torch.cuda.is_available() else -1
print("CUDA:", torch.cuda.is_available(), "| device:", DEVICE)

RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

DEV_JSON = ROOT / "data" / "processed" / "dev.json"
MAX_LEN  = 384
STRIDE   = 128
print("transformers:", transformers.__version__)
print("OK:", tiktoken.__version__)

CUDA: False | device: -1
transformers: 5.18.0
OK: 0.14.0


In [3]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

dev_raw = load_jsonl(DEV_JSON)
print(f"dev = {len(dev_raw)}")
print("keys:", list(dev_raw[0].keys()))
print("sample:", {k: dev_raw[0][k] for k in list(dev_raw[0].keys())[:6]})

dev = 3814
keys: ['id', 'uit_id', 'title', 'context', 'question', 'answers', 'is_impossible', 'plausible_answers']
sample: {'id': '0001-0001-0001', 'uit_id': 'uit_000001', 'title': 'Paris', 'context': 'Paris nằm ở điểm gặp nhau của các hành trình thương mại đường bộ và đường sông, và là trung tâm của một vùng nông nghiệp giàu có. Vào thế kỷ 10, Paris đã là một trong những thành phố chính của Pháp cùng các cung điện hoàng gia, các tu viện và nhà thờ. Từ thế kỷ 12, Paris trở thành một trong những trung tâm của châu Âu về giáo dục và nghệ thuật. Thế kỷ 14, Paris là thành phố quan trọng bậc nhất của Cơ Đốc giáo và trong các thế kỷ 16, 17, đây là nơi diễn ra Cách mạng Pháp cùng nhiều sự kiện lịch sử quan trọng của Pháp và châu Âu. Đến thế kỷ 19 và 20, thành phố trở thành một trong những trung tâm văn hóa của thế giới, thủ đô của nghệ thuật và giải trí.', 'question': 'Paris đạt được thành quả gì sau khoảng 4 thế kỷ tính từ ngày Cách mạng Pháp diễn ra?', 'answers': {'text': ['trở thành một tr

In [4]:
def to_squad_example(row, idx):
    answers = row.get("answers", {})
    if isinstance(answers, dict):
        texts = answers.get("text", []) or []
        starts = answers.get("answer_start", []) or []
    else:
        texts = answers if isinstance(answers, list) else []
        starts = [row["context"].find(t) for t in texts]

    is_imp = bool(row.get("is_impossible", len(texts) == 0))
    if is_imp:
        texts, starts = [], []

    return {
        "id":        str(row.get("id", row.get("uit_id", idx))),
        "question":  row["question"],
        "context":   row["context"],
        "answers":   {"text": texts, "answer_start": starts},
        "is_impossible": is_imp,
    }

dev_examples = [to_squad_example(r, i) for i, r in enumerate(dev_raw)]

n_noans = sum(1 for e in dev_examples if e["is_impossible"])
print(f"dev={len(dev_examples)}  no-answer={n_noans}  "
      f"answerable={len(dev_examples)-n_noans}")

dev=3814  no-answer=1161  answerable=2653


In [5]:
def normalize_text(s):
    s = unicodedata.normalize("NFC", s).lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    return " ".join(s.split())

def exact_match(pred, golds):
    return float(any(normalize_text(pred) == normalize_text(g) for g in golds))

def token_f1(pred, golds):
    pt = normalize_text(pred).split()
    best = 0.0
    for g in golds:
        gt = normalize_text(g).split()
        if not pt and not gt:
            best = max(best, 1.0); continue
        if not pt or not gt:
            continue
        common = Counter(pt) & Counter(gt)
        same = sum(common.values())
        if same == 0:
            continue
        p = same / len(pt)
        r = same / len(gt)
        best = max(best, 2 * p * r / (p + r))
    return best

def compute_squad20_metrics(predictions, references):
    ref_by_id = {r["id"]: r for r in references}
    em_has, f1_has, em_no, f1_no = [], [], [], []
    for p in predictions:
        r = ref_by_id.get(p["id"])
        if r is None:
            continue
        pred = p["prediction_text"] or ""
        if r["is_impossible"]:
            correct = float(pred.strip() == "")
            em_no.append(correct); f1_no.append(correct)
        else:
            golds = r["answers"]["text"]
            em_has.append(exact_match(pred, golds))
            f1_has.append(token_f1(pred, golds))

    def _avg(x): return float(np.mean(x) * 100) if x else 0.0

    n_has, n_no = len(em_has), len(em_no)
    total = n_has + n_no
    em_overall = (_avg(em_has) * n_has + _avg(em_no) * n_no) / total if total else 0.0
    f1_overall = (_avg(f1_has) * n_has + _avg(f1_no) * n_no) / total if total else 0.0

    return {
        "EM_HasAns": _avg(em_has), "F1_HasAns": _avg(f1_has),
        "EM_NoAns":  _avg(em_no),  "F1_NoAns":  _avg(f1_no),
        "EM_overall": em_overall,  "F1_overall": f1_overall,
        "n_HasAns": n_has, "n_NoAns": n_no,
    }

In [6]:
import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

# Model weights: lấy từ deepset (bản gốc)
MODEL_NAME = "deepset/xlm-roberta-base-squad2"

# Tokenizer: lấy từ repo có sẵn tokenizer.json (tương thích với transformers 5.x)
TOKENIZER_NAME = "helenai/deepset-xlm-roberta-base-squad2-ov"

zs_tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, use_fast=True)
zs_model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

zs_model.eval()
if DEVICE >= 0:
    zs_model = zs_model.to(f"cuda:{DEVICE}")

MAX_LEN = 384
STRIDE  = 128
SAMPLE_SIZE = 0  # 0 = full 3814, đặt 200 để test nhanh
dev_eval = dev_examples[:SAMPLE_SIZE] if SAMPLE_SIZE else dev_examples
print(f"eval on {len(dev_eval)} samples | device={DEVICE}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

eval on 3814 samples | device=-1


In [7]:
import numpy as np

@torch.no_grad()
def answer_question(question, context,
                    model, tokenizer,
                    max_len=384, stride=128,
                    n_best=20, max_ans_len=30,
                    device=-1):
    """
    Trả về (predicted_text, score). predicted_text == "" nghĩa là no-answer.
    """
    enc = tokenizer(
        question, context,
        max_length=max_len,
        truncation="only_second",
        stride=stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
        return_tensors="pt",
    )

    offsets = enc.pop("offset_mapping")       # (n_windows, seq_len, 2)
    enc.pop("overflow_to_sample_mapping", None)

    tgt = f"cuda:{device}" if device >= 0 else "cpu"
    enc = {k: v.to(tgt) for k, v in enc.items() if k != "token_type_ids"}

    out = model(**enc)
    start_logits = out.start_logits.cpu().numpy()
    end_logits   = out.end_logits.cpu().numpy()

    ctx_len = len(context)
    best_text, best_score = "", -1e9
    cls_score = -1e9

    for i in range(start_logits.shape[0]):
        s_log = start_logits[i]
        e_log = end_logits[i]
        off   = offsets[i].tolist()

        valid_pos = [
            j for j, o in enumerate(off)
            if (o[0] != 0 or o[1] != 0) and o[0] < ctx_len and o[1] <= ctx_len
        ]

        cls_score = max(cls_score, float(s_log[0] + e_log[0]))
        if not valid_pos:
            continue

        s_idx = sorted(valid_pos, key=lambda j: s_log[j], reverse=True)[:n_best]
        e_idx = sorted(valid_pos, key=lambda j: e_log[j], reverse=True)[:n_best]

        for si in s_idx:
            for ei in e_idx:
                if ei < si or ei - si + 1 > max_ans_len:
                    continue
                score = float(s_log[si] + e_log[ei])
                if score > best_score:
                    best_score = score
                    s_char = off[si][0]
                    e_char = off[ei][1]
                    if 0 <= s_char < e_char <= ctx_len:
                        best_text = context[s_char:e_char]
    DELTA = 2.0
    if best_score < cls_score + DELTA:
        return "", best_score
    return best_text, best_score


# ---- Chạy inference ----
predictions = []
for i, ex in enumerate(dev_eval):
    try:
        pred, _ = answer_question(
            ex["question"], ex["context"],
            zs_model, zs_tokenizer,
            max_len=MAX_LEN, stride=STRIDE,
            device=DEVICE,
        )
    except Exception as e:
        pred = ""
    predictions.append({"id": ex["id"], "prediction_text": pred})

    if (i + 1) % 100 == 0:
        print(f"  {i+1}/{len(dev_eval)}")
print("done:", len(predictions))

  100/3814
  200/3814
  300/3814
  400/3814
  500/3814
  600/3814
  700/3814
  800/3814
  900/3814
  1000/3814
  1100/3814
  1200/3814
  1300/3814
  1400/3814
  1500/3814
  1600/3814
  1700/3814
  1800/3814
  1900/3814
  2000/3814
  2100/3814
  2200/3814
  2300/3814
  2400/3814
  2500/3814
  2600/3814
  2700/3814
  2800/3814
  2900/3814
  3000/3814
  3100/3814
  3200/3814
  3300/3814
  3400/3814
  3500/3814
  3600/3814
  3700/3814
  3800/3814
done: 3814


In [8]:
references = [
    {"id": e["id"], "answers": e["answers"], "is_impossible": e["is_impossible"]}
    for e in dev_eval
]

zs_metrics = compute_squad20_metrics(predictions, references)
zs_metrics.update({
    "model":       MODEL_NAME,        # deepset/xlm-roberta-base-squad2
    "tokenizer":   TOKENIZER_NAME,    # helenai/...-ov
    "mode":        "zero-shot",
    "dataset":     "UIT-ViQuAD2.0",
    "sample_size": len(dev_eval),
    "max_len":     MAX_LEN,
    "stride":      STRIDE,
})

with open(RESULTS_DIR / "zeroshot_predictions.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
with open(RESULTS_DIR / "zeroshot_metrics.json", "w", encoding="utf-8") as f:
    json.dump(zs_metrics, f, ensure_ascii=False, indent=2)

print(json.dumps(zs_metrics, ensure_ascii=False, indent=2))

{
  "EM_HasAns": 39.65322276667923,
  "F1_HasAns": 58.754459768485304,
  "EM_NoAns": 45.65030146425496,
  "F1_NoAns": 45.65030146425496,
  "EM_overall": 41.47876245411641,
  "F1_overall": 54.76549076187507,
  "n_HasAns": 2653,
  "n_NoAns": 1161,
  "model": "deepset/xlm-roberta-base-squad2",
  "tokenizer": "helenai/deepset-xlm-roberta-base-squad2-ov",
  "mode": "zero-shot",
  "dataset": "UIT-ViQuAD2.0",
  "sample_size": 3814,
  "max_len": 384,
  "stride": 128
}


In [9]:
for i in range(min(10, len(predictions))):
    p = predictions[i]
    e = dev_eval[i]
    print(f"\n[{i}] Q: {e['question']}")
    print(f"    Context: {e['context'][:140]}...")
    if e["is_impossible"]:
        print(f"    GOLD:  <no answer>")
    else:
        print(f"    GOLD:  {e['answers']['text']}")
    pred_display = p['prediction_text'] if p['prediction_text'] else "<no answer>"
    print(f"    PRED:  {pred_display}")


[0] Q: Paris đạt được thành quả gì sau khoảng 4 thế kỷ tính từ ngày Cách mạng Pháp diễn ra?
    Context: Paris nằm ở điểm gặp nhau của các hành trình thương mại đường bộ và đường sông, và là trung tâm của một vùng nông nghiệp giàu có. Vào thế kỷ...
    GOLD:  ['trở thành một trong những trung tâm văn hóa của thế giới, thủ đô của nghệ thuật và giải trí']
    PRED:  <no answer>

[1] Q: Vị trí địa lý của Pháp có gì đặc biệt?
    Context: Paris nằm ở điểm gặp nhau của các hành trình thương mại đường bộ và đường sông, và là trung tâm của một vùng nông nghiệp giàu có. Vào thế kỷ...
    GOLD:  <no answer>
    PRED:  <no answer>

[2] Q: Kinh tế xung quanh kinh đô ánh sáng mạnh về gì?
    Context: Paris nằm ở điểm gặp nhau của các hành trình thương mại đường bộ và đường sông, và là trung tâm của một vùng nông nghiệp giàu có. Vào thế kỷ...
    GOLD:  ['giáo dục và nghệ thuật', 'nông nghiệp']
    PRED:  <no answer>

[3] Q: Cuộc cách mạng Pháp diễn ra ở đâu?
    Context: Paris nằm ở điểm gặp nhau

In [10]:
n_pred_noans = sum(1 for p in predictions if not p["prediction_text"].strip())
n_gold_noans = sum(1 for e in dev_eval if e["is_impossible"])

print(f"Gold no-answer:  {n_gold_noans}/{len(dev_eval)} "
      f"({100*n_gold_noans/len(dev_eval):.1f}%)")
print(f"Pred no-answer:  {n_pred_noans}/{len(predictions)} "
      f"({100*n_pred_noans/len(predictions):.1f}%)")

tp = sum(1 for p, e in zip(predictions, dev_eval)
         if e["is_impossible"] and not p["prediction_text"].strip())
fp = sum(1 for p, e in zip(predictions, dev_eval)
         if not e["is_impossible"] and not p["prediction_text"].strip())
fn = sum(1 for p, e in zip(predictions, dev_eval)
         if e["is_impossible"] and p["prediction_text"].strip())

print(f"\nNo-answer confusion:")
print(f"  TP (gold=no,  pred=no):  {tp}")
print(f"  FP (gold=yes, pred=no):  {fp}   từ chối khi có đáp án")
print(f"  FN (gold=no,  pred=yes): {fn}   đoán bừa khi không có đáp án")

Gold no-answer:  1161/3814 (30.4%)
Pred no-answer:  1097/3814 (28.8%)

No-answer confusion:
  TP (gold=no,  pred=no):  530
  FP (gold=yes, pred=no):  567   từ chối khi có đáp án
  FN (gold=no,  pred=yes): 631   đoán bừa khi không có đáp án


VNAI/Phobert

In [11]:
import inspect
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.append(str(ROOT / "src"))

from data_preprocessing import (
    prepare_train_features,
    prepare_validation_features,
)

print("prepare_train_features:")
print("  ", inspect.signature(prepare_train_features))
print("\nprepare_validation_features:")
print("  ", inspect.signature(prepare_validation_features))

print("\nDocstring prepare_train_features:")
print(inspect.getdoc(prepare_train_features) or "(không có)")

prepare_train_features:
   (examples: Dict[str, Any], tokenizer: Any, max_seq_length: int = 384, doc_stride: int = 128) -> Dict[str, Any]

prepare_validation_features:
   (examples: Dict[str, Any], tokenizer: Any, max_seq_length: int = 384, doc_stride: int = 128) -> Dict[str, Any]

Docstring prepare_train_features:
Tiền xử lý tập Train: Tokenize, trượt cửa sổ và ánh xạ span từ ký tự sang token.


In [12]:
import json, random
from datasets import Dataset

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

TRAIN_JSON = ROOT / "data" / "processed" / "train.json"
train_raw = load_jsonl(TRAIN_JSON)
print(f"train = {len(train_raw)}  dev = {len(dev_raw)}")
TRAIN_SAMPLE = 200
random.seed(42)
if TRAIN_SAMPLE and TRAIN_SAMPLE < len(train_raw):
    idx = random.sample(range(len(train_raw)), TRAIN_SAMPLE)
    train_raw = [train_raw[i] for i in idx]
    print(f"→ subsample train = {len(train_raw)}")

train_examples = [to_squad_example(r, i) for i, r in enumerate(train_raw)]

n_tr_noans = sum(1 for e in train_examples if e["is_impossible"])
print(f"train no-answer = {n_tr_noans} ({100*n_tr_noans/len(train_examples):.1f}%)")

train = 8000  dev = 3814
→ subsample train = 200
train no-answer = 61 (30.5%)


In [13]:
FT_MODEL = "bert-base-multilingual-cased"
# FT_MODEL = "vinai/phobert-base-v2"
is_phobert = "phobert" in FT_MODEL.lower()
FT_MAX_LEN = 256 if is_phobert else 384
STRIDE     = 128

from transformers import AutoTokenizer, AutoModelForQuestionAnswering

ft_tokenizer = AutoTokenizer.from_pretrained(FT_MODEL, use_fast=True)
print("tokenizer:", type(ft_tokenizer).__name__)

# Test nhanh
test_ids = ft_tokenizer.encode("Xin chào", add_special_tokens=False)
print("test ids:", test_ids[:10])
print("test decode:", ft_tokenizer.decode(test_ids[:10]))
print("max_len:", ft_tokenizer.model_max_length)

tokenizer: BertTokenizer
test ids: [59876, 10115, 18643, 44595]
test decode: Xin chào
max_len: 512


In [14]:
sample_batch = {
    "id":        [train_examples[0]["id"], train_examples[1]["id"]],
    "question":  [train_examples[0]["question"], train_examples[1]["question"]],
    "context":   [train_examples[0]["context"], train_examples[1]["context"]],
    "answers":   [train_examples[0]["answers"], train_examples[1]["answers"]],
    "is_impossible": [train_examples[0]["is_impossible"], train_examples[1]["is_impossible"]],
}

try:
    test_out = prepare_train_features(
        sample_batch,
        tokenizer=ft_tokenizer,
        max_seq_length=FT_MAX_LEN,
        doc_stride=STRIDE,
    )
    print("prepare_train_features gọi thành công")
    print("keys:", list(test_out.keys()))
    print("n_windows:", len(test_out["input_ids"]))
    print("start_positions:", test_out["start_positions"])
    print("end_positions:", test_out["end_positions"])
except TypeError as e:
    print("Signature khác giả định:", e)
    print("Xem Cell P1 và điều chỉnh tên tham số")

prepare_train_features gọi thành công
keys: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions']
n_windows: 2
start_positions: [157, 0]
end_positions: [169, 0]


In [15]:
test_val = prepare_validation_features(
    sample_batch,
    tokenizer=ft_tokenizer,
    max_seq_length=FT_MAX_LEN,
    doc_stride=STRIDE,
)
print("keys:", list(test_val.keys()))
for k, v in test_val.items():
    print(f"  {k:20s} → {type(v).__name__}, len={len(v) if hasattr(v,'__len__') else '-'}")

keys: ['input_ids', 'token_type_ids', 'attention_mask', 'offset_mapping', 'example_id']
  input_ids            → list, len=2
  token_type_ids       → list, len=2
  attention_mask       → list, len=2
  offset_mapping       → list, len=2
  example_id           → list, len=2


In [16]:
from datasets import Dataset

train_ds = Dataset.from_list(train_examples)
# dev_ds đã có từ Task 1
if "dev_ds" not in dir():
    dev_ds = Dataset.from_list(dev_examples)

# --- Train: chỉ giữ 5 cột cần cho Trainer ---
KEEP_TRAIN = ["input_ids", "attention_mask", "token_type_ids",
              "start_positions", "end_positions"]

def prep_train_wrapper(examples):
    return prepare_train_features(
        examples,
        tokenizer=ft_tokenizer,
        max_seq_length=FT_MAX_LEN,
        doc_stride=STRIDE,
    )

# detect columns thực tế từ 1 batch nhỏ
_probe = prep_train_wrapper(train_ds[:2])
_keep_train = [c for c in KEEP_TRAIN if c in _probe]
_remove_train = [c for c in train_ds.column_names if c not in _keep_train]

train_features = train_ds.map(
    prep_train_wrapper, batched=True, remove_columns=_remove_train,
)
print(f"train features: {len(train_features)} windows")
print(f"columns: {train_features.column_names}")

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

train features: 216 windows
columns: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions']


In [19]:
from transformers import (
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from pathlib import Path

ft_model = AutoModelForQuestionAnswering.from_pretrained(FT_MODEL)

OUT_DIR = ROOT / "models" / FT_MODEL.replace("/", "_") / "viquad2_demo"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Cấu hình cho CPU
CPU_MODE   = not torch.cuda.is_available()
EPOCHS     = 1
BATCH_SIZE = 8 if CPU_MODE else 16
GRAD_ACC   = 4 if CPU_MODE else 2

args = TrainingArguments(
    output_dir=str(OUT_DIR),
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC,
    max_steps=50,                 
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="no",           
    save_strategy="no",
    logging_steps=10,
    fp16=False,
    bf16=False,
    seed=42,
    report_to="none",
)

trainer = Trainer(
    model=ft_model,
    args=args,
    train_dataset=train_features,
    processing_class=ft_tokenizer,      
    data_collator=DataCollatorWithPadding(ft_tokenizer),
)

print(f"Bắt đầu train: model={FT_MODEL}")
print(f"  samples={len(train_features)}  batch={BATCH_SIZE}×{GRAD_ACC}  max_steps=50")
print(f"  device={'cpu' if CPU_MODE else 'cuda'}")

import time
t0 = time.time()
trainer.train()
print(f"\nTrain xong sau {(time.time()-t0)/60:.1f} phút")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

Bắt đầu train: model=bert-base-multilingual-cased
  samples=216  batch=8×4  max_steps=50
  device=cpu


d:\uit-viquad2-extractive-qa\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
10,5.539967
20,4.545516
30,3.581201
40,2.867484
50,2.414787



Train xong sau 30.7 phút


In [21]:
from datasets import Dataset

def process_one(ex):
    out = prepare_validation_features(
        {
            "id":            [ex["id"]],
            "question":      [ex["question"]],
            "context":       [ex["context"]],
            "answers":       [ex["answers"]],
            "is_impossible": [ex["is_impossible"]],
        },
        tokenizer=ft_tokenizer,
        max_seq_length=FT_MAX_LEN,
        doc_stride=STRIDE,
    )
    n = len(out["input_ids"])
    return {
        "input_ids":      out["input_ids"],
        "attention_mask": out["attention_mask"],
        "token_type_ids": out.get("token_type_ids", [[0] * len(out["input_ids"][0])] * n),
        "offset_mapping": out["offset_mapping"],
        "example_id":     [ex["id"]] * n,           # ← expand theo n windows
    }

# Build val_features bằng cách gọi từng example
val_dict = {
    "input_ids":      [],
    "attention_mask": [],
    "token_type_ids": [],
    "offset_mapping": [],
    "example_id":     [],
}

for i, ex in enumerate(dev_examples):
    r = process_one(ex)
    for k in val_dict:
        val_dict[k].extend(r[k])
    if (i + 1) % 500 == 0:
        print(f"  {i+1}/{len(dev_examples)}")

val_features = Dataset.from_dict(val_dict)
print(f"\nval_features: {len(val_features)} windows")
print("columns:", val_features.column_names)

  500/3814
  1000/3814
  1500/3814
  2000/3814
  2500/3814
  3000/3814
  3500/3814

val_features: 3928 windows
columns: ['input_ids', 'attention_mask', 'token_type_ids', 'offset_mapping', 'example_id']


In [22]:
eval_cols = ["input_ids", "attention_mask", "token_type_ids"]
val_for_model = val_features.remove_columns(
    [c for c in val_features.column_names if c not in eval_cols]
)

predictions_raw = trainer.predict(val_for_model)
start_logits = predictions_raw.predictions[0]
end_logits   = predictions_raw.predictions[1]
print("logits shape:", start_logits.shape)

logits shape: (3928, 384)


In [24]:
from collections import defaultdict

def postprocess_squad(examples, features, start_logits, end_logits,
                      n_best=20, max_ans_len=30):
    key = "example_id" if "example_id" in features.column_names else "id"
    id_to_idx = {str(ex["id"]): i for i, ex in enumerate(examples)}

    feat_per_example = defaultdict(list)
    for i, feat in enumerate(features):
        eid = str(feat[key])
        if eid in id_to_idx:
            feat_per_example[id_to_idx[eid]].append(i)

    preds = []
    for ex_idx, ex in enumerate(examples):
        fidxs = feat_per_example.get(ex_idx, [])
        ctx = ex["context"]
        ctx_len = len(ctx)
        best_text, best_score = "", -1e9
        cls_score = -1e9

        for fi in fidxs:
            s_log = start_logits[fi]
            e_log = end_logits[fi]
            off = [list(o) if o is not None else [0, 0]
                   for o in features[fi]["offset_mapping"]]

            valid_pos = [j for j, o in enumerate(off)
                         if (o[0] != 0 or o[1] != 0) and 0 <= o[0] < o[1] <= ctx_len]

            cls_score = max(cls_score, float(s_log[0] + e_log[0]))
            if not valid_pos:
                continue

            s_idx = sorted(valid_pos, key=lambda j: s_log[j], reverse=True)[:n_best]
            e_idx = sorted(valid_pos, key=lambda j: e_log[j], reverse=True)[:n_best]

            for si in s_idx:
                for ei in e_idx:
                    if ei < si or ei - si + 1 > max_ans_len:
                        continue
                    score = float(s_log[si] + e_log[ei])
                    if score > best_score:
                        best_score = score
                        best_text = ctx[off[si][0]:off[ei][1]]

        pred = best_text if best_score >= cls_score + 2.0 else ""
        preds.append({"id": ex["id"], "prediction_text": pred})

    return preds


ft_predictions = postprocess_squad(
    examples=dev_examples,
    features=val_features,
    start_logits=start_logits,
    end_logits=end_logits,
)
print(f"predictions: {len(ft_predictions)}")

predictions: 3814


In [25]:
finetuned_metrics = compute_squad20_metrics(ft_predictions, references)
finetuned_metrics.update({
    "model":           FT_MODEL,
    "mode":            "fine-tuned",
    "dataset":         "UIT-ViQuAD2.0",
    "epochs":          EPOCHS,
    "effective_batch": BATCH_SIZE * GRAD_ACC,
    "train_size":      len(train_features),
    "seed":            42,
    "delta":           2.0,
})

print("=== FINE-TUNED ===")
print(json.dumps(finetuned_metrics, ensure_ascii=False, indent=2))

import pandas as pd
compare = pd.DataFrame([
    {"mode": "Zero-shot",  "model": MODEL_NAME, **zs_metrics},      
    {"mode": "Fine-tuned", "model": FT_MODEL,   **finetuned_metrics},
])
cols = ["mode", "model", "EM_HasAns", "F1_HasAns",
        "EM_NoAns", "F1_NoAns", "EM_overall", "F1_overall"]
print("\n=== SO SÁNH ===")
print(compare[cols].to_string(index=False))

=== FINE-TUNED ===
{
  "EM_HasAns": 0.22615906520919715,
  "F1_HasAns": 0.41563671712262185,
  "EM_NoAns": 99.31093884582258,
  "F1_NoAns": 99.31093884582258,
  "EM_overall": 30.38804404824332,
  "F1_overall": 30.51984378881131,
  "n_HasAns": 2653,
  "n_NoAns": 1161,
  "model": "bert-base-multilingual-cased",
  "mode": "fine-tuned",
  "dataset": "UIT-ViQuAD2.0",
  "epochs": 1,
  "effective_batch": 32,
  "train_size": 216,
  "seed": 42,
  "delta": 2.0
}

=== SO SÁNH ===
      mode                           model  EM_HasAns  F1_HasAns  EM_NoAns  F1_NoAns  EM_overall  F1_overall
 zero-shot deepset/xlm-roberta-base-squad2  39.653223  58.754460 45.650301 45.650301   41.478762   54.765491
fine-tuned    bert-base-multilingual-cased   0.226159   0.415637 99.310939 99.310939   30.388044   30.519844


In [26]:
with open(RESULTS_DIR / f"finetuned_{FT_MODEL.replace('/', '_')}_predictions.json",
          "w", encoding="utf-8") as f:
    json.dump(ft_predictions, f, ensure_ascii=False, indent=2)
with open(RESULTS_DIR / f"finetuned_{FT_MODEL.replace('/', '_')}_metrics.json",
          "w", encoding="utf-8") as f:
    json.dump(finetuned_metrics, f, ensure_ascii=False, indent=2)

LOG_DIR = RESULTS_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)
from datetime import datetime
ts = datetime.now().strftime("%Y%m%d_%H%M%S")

(LOG_DIR / f"finetuned_{FT_MODEL.replace('/', '_')}_{ts}.json").write_text(
    json.dumps({
        "experiment": "Fine-tuned " + FT_MODEL,
        "config": {
            "model":      FT_MODEL,
            "max_len":    FT_MAX_LEN,
            "stride":     STRIDE,
            "epochs":     EPOCHS,
            "batch":      BATCH_SIZE,
            "grad_accum": GRAD_ACC,
            "lr":         2e-5,
            "seed":       42,
            "delta":      2.0,
            "device":     "cpu" if CPU_MODE else "cuda",
            "train_size": len(train_features),
        },
        "metrics": finetuned_metrics,
    }, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("Đã lưu log")
print("   →", RESULTS_DIR / f"finetuned_{FT_MODEL.replace('/', '_')}_metrics.json")
print("   →", LOG_DIR / f"finetuned_{FT_MODEL.replace('/', '_')}_{ts}.json")

Đã lưu log
   → d:\uit-viquad2-extractive-qa\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\finetuned_bert-base-multilingual-cased_metrics.json
   → d:\uit-viquad2-extractive-qa\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\results\logs\finetuned_bert-base-multilingual-cased_20261006_220709.json
